In [6]:
import pandas as pd
import fsspec
from zipfile import ZipFile
from io import BytesIO

def load_specific_csv_from_zip(url, filename):
    """Load one specific CSV from multi-file ZIP"""
    with fsspec.open(url, 'rb') as f:
        with ZipFile(BytesIO(f.read())) as z:
            return pd.read_csv(z.open(filename))

url = "gs://agntworks-data-dev/wheelsup/raw/wingx/WingX_2025.zip"

# Create quarter-wise DataFrames (exactly matching your file names)
df_2025_Q1 = load_specific_csv_from_zip(url, 'WINGX_Jan25-Mar25.csv')
df_2025_Q2 = load_specific_csv_from_zip(url, 'WINGX_Apr25-Jun25.csv')
df_2025_Q3 = load_specific_csv_from_zip(url, 'WINGX_Jul25-Sep25.csv')
df_2025_Q4 = load_specific_csv_from_zip(url, 'WINGX_Oct25-Dec25.csv')

# Combine ALL quarters into master df
df = pd.concat([df_2025_Q1, df_2025_Q2, df_2025_Q3, df_2025_Q4], ignore_index=True)

# Verify
print("Q1 shape:", df_2025_Q1.shape)
print("Q2 shape:", df_2025_Q2.shape) 
print("Q3 shape:", df_2025_Q3.shape)
print("Q4 shape:", df_2025_Q4.shape)
print("FULL df shape:", df.shape)
print("\nFirst few rows of combined df:")
df.head()

/var/tmp/ipykernel_32764/2728868874.py:10: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(z.open(filename))


Q1 shape: (841121, 18)
Q2 shape: (917566, 18)
Q3 shape: (939306, 18)
Q4 shape: (938784, 18)
FULL df shape: (3636777, 18)

First few rows of combined df:


,FlightDate_utc,ArrivalDate_utc,Operator,FromAirport,FromCity,FromState,ToAirport,ToCity,ToState,Hours,aircraft_tailsign,aircraft_tailsign_certification,operator_type,aircraft_icao_code,aircraft_type,aircraft_model,aircraft_segment,fuel_uplift_usg
0,2025-03-27T18:06:00.000Z,2025-03-27T19:54:00.000Z,Conquest Express LLC,KRIC,Richmond,VA,KENW,Kenosha,WI,1.800000,N341AE,Part 91 / Non Commercial,Corporate Flight Department,LJ40,Learjet 40,Learjet 40-XR,Super Light Jet,363.60
1,2025-03-18T12:51:00.000Z,2025-03-18T13:40:00.000Z,MLA Management LLC,KSLC,Salt Lake City,UT,KRIW,Riverton,WY,0.816667,N341AR,Part 91 / Non Commercial,Corporate Flight Department,C525,Cessna-Citation CJ1 / CitationJet / 525,Citation CJ1,Entry Level Jet,109.43
2,2025-01-26T20:28:00.000Z,2025-01-26T22:07:00.000Z,341HB LLC,KEDC,Austin,TX,KTVI,Thomasville,GA,1.650000,N341HB,Part 91 / Non Commercial,Corporate Flight Department,LJ45,Learjet 45,Learjet 45,Super Light Jet,331.65
3,2025-02-16T21:16:00.000Z,2025-02-16T21:54:00.000Z,341HB LLC,KBKS,Falfurrias,TX,KEDC,Austin,TX,0.633333,N341HB,Part 91 / Non Commercial,Corporate Flight Department,LJ45,Learjet 45,Learjet 45,Super Light Jet,127.30
4,2025-03-19T10:10:00.000Z,2025-03-19T12:11:00.000Z,341HB LLC,KEDC,Austin,TX,KSRQ,Sarasota Bradenton,FL,2.016667,N341HB,Part 91 / Non Commercial,Corporate Flight Department,LJ45,Learjet 45,Learjet 45,Super Light Jet,405.35


In [9]:
# 1. Check for Missing Values (Nulls)
null_counts = df.isnull().sum()

# 2. Check for Unique Values (Cardinality)
unique_counts = df.nunique()

# 3. Calculate the % of uniqueness (Ratio)
# If this ratio is very low (e.g. < 5%), it's a great candidate for 'category'
unique_ratio = (df.nunique() / len(df)) * 100

# Combine into a summary table
eda_summary = pd.DataFrame({
    'Null Values': null_counts,
    'Unique Values': unique_counts,
    'Uniqueness Ratio (%)': unique_ratio.round(4)
})

print(eda_summary)

                                 Null Values  Unique Values  \
FlightDate_utc                             0         888810   
ArrivalDate_utc                            0         901747   
Operator                                   0          14898   
FromAirport                                0           9276   
FromCity                               43673           5429   
FromState                             763282            325   
ToAirport                                  0           9959   
ToCity                                 47442           5648   
ToState                               766258            328   
Hours                                      0          28760   
aircraft_tailsign                          0          24809   
aircraft_tailsign_certification            0              3   
operator_type                              0              6   
aircraft_icao_code                         0            115   
aircraft_type                              0           

In [10]:
# 'K' is the prefix for the Continental USA. 
# Anything else (C=Canada, L=Europe, M=Mexico) is International.
non_us_prefixes = df[df['ToState'].isnull()]['ToAirport'].str[0].value_counts()
print("Top 10 Airport Prefixes for Missing States:")
print(non_us_prefixes.head(10))

Top 10 Airport Prefixes for Missing States:
ToAirport
L    351295
E    179582
M     51187
S     34928
K     23978
T     22446
O     20160
R     16576
V     15085
G     10867
Name: count, dtype: int64


In [11]:
# 1. Filter for FromAirport starting with 'K' AND FromState being Null
k_null_from = df[(df['FromAirport'].str.startswith('K')) & (df['FromState'].isna())]

# 2. Get the top 10 most frequent mystery airports
top_10_k_nulls = k_null_from['FromAirport'].value_counts().head(10)

print("Top 10 US Airports (K-prefix) with missing State/City data:")
print(top_10_k_nulls)

Top 10 US Airports (K-prefix) with missing State/City data:
FromAirport
KT82    1465
KF45     971
K27K     872
K5B2     718
K0A9     636
KX60     535
K1A5     504
K11R     433
K1R8     426
K4A9     409
Name: count, dtype: int64


In [12]:
import pandas as pd

# 1. First, handle the Nulls so they are labeled correctly in the lookup
df['FromState'] = df['FromState'].fillna('International/Unknown')
df['ToState'] = df['ToState'].fillna('International/Unknown')
df['FromCity'] = df['FromCity'].fillna('Unknown')
df['ToCity'] = df['ToCity'].fillna('Unknown')

# 2. Convert Dates (This turns strings into math-friendly objects)
df['FlightDate_utc'] = pd.to_datetime(df['FlightDate_utc'])
df['ArrivalDate_utc'] = pd.to_datetime(df['ArrivalDate_utc'])

# 3. Apply the Category Mapping (The "Lookup" Magic)
# These are the columns where we replace bulk text with short reference codes
cat_columns = [
    'Operator', 'FromAirport', 'FromCity', 'FromState', 
    'ToAirport', 'ToCity', 'ToState', 'aircraft_tailsign',
    'aircraft_tailsign_certification', 'operator_type', 
    'aircraft_icao_code', 'aircraft_type', 'aircraft_model', 'aircraft_segment'
]

for col in cat_columns:
    df[col] = df[col].astype('category')

# 4. Downcast numbers (Saves 50% space on these columns)
df['Hours'] = pd.to_numeric(df['Hours'], downcast='float')
df['fuel_uplift_usg'] = pd.to_numeric(df['fuel_uplift_usg'], downcast='float')

# 5. The Moment of Truth: Check how much memory you saved
print(df.info(memory_usage='deep'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3636777 entries, 0 to 3636776
Data columns (total 18 columns):
 #   Column                           Dtype              
---  ------                           -----              
 0   FlightDate_utc                   datetime64[ns, UTC]
 1   ArrivalDate_utc                  datetime64[ns, UTC]
 2   Operator                         category           
 3   FromAirport                      category           
 4   FromCity                         category           
 5   FromState                        category           
 6   ToAirport                        category           
 7   ToCity                           category           
 8   ToState                          category           
 9   Hours                            float32            
 10  aircraft_tailsign                category           
 11  aircraft_tailsign_certification  category           
 12  operator_type                    category           
 13  aircraft_ica

In [13]:
# Returns an array of unique names
print(df['Operator'].unique())

['Conquest Express LLC', 'MLA Management LLC', '341HB LLC', 'G3 Aviation LLC', 'N342ZK-Operator', ..., 'TD Aviation LLC', 'VTMIR-Operator', 'N929-Operator', 'Air Mauritius', 'OHDAM-Operator']
Length: 14898
Categories (14898, object): ['01941-Operator', '01949-Operator', '0262 Aviation Inc Trustee', '09307-Operator', ..., 'flyNEAT', 'flydubai', 'iXAir', 'nextair LLC']


In [14]:
# 1. Get flight counts for every operator
counts = df['Operator'].value_counts()

# 2. Isolate the Top 20
top_20 = counts.head(20).to_frame(name='Flight Count')

# 3. Sum up all operators from index 20 onwards
others_sum = counts.iloc[20:].sum()

# 4. Create a row for the 'Other' category
# We subtract 20 from the total length to show exactly how many operators are grouped
other_label = f"Other Operators ({len(counts) - 20:,} total)"
other_row = pd.DataFrame({'Flight Count': [others_sum]}, index=[other_label])

# 5. Combine Top 20 + Other
market_summary = pd.concat([top_20, other_row])

# 6. Calculate Market Share Percentage
total_flights = market_summary['Flight Count'].sum()
market_summary['Market Share %'] = (market_summary['Flight Count'] / total_flights * 100).round(2)

print(market_summary)

                                Flight Count  Market Share %
NetJets                               432787           11.90
Flexjet                               174109            4.79
NetJets Europe                         66648            1.83
flyExclusive                           44247            1.22
Wheels Up Private Jets                 41508            1.14
Executive Jet Management               39434            1.08
VistaJet Ltd                           38870            1.07
Vista America                          37108            1.02
Solairus Aviation                      32755            0.90
Contour Aviation                       28709            0.79
Jet Linx Aviation                      24122            0.66
Airsprint Inc                          23625            0.65
Vista Germany                          17589            0.48
Nicholas Air                           16766            0.46
Executive AirShare                     14875            0.41
Baker Aviation LLC      

In [24]:
url = "https://raw.githubusercontent.com/agntworks-ai/agntworks-experiments/refs/heads/main/pricing/eda/amit/icao_cluster.csv?token=GHSAT0AAAAAADZDKUWQIFTRVHS5DBIH5JNY2PHGHQQ"
df_cluster = pd.read_csv(url)
print("Shape:", df_cluster.shape)
print("\nHead:")
df_cluster.head()

Shape: (38851, 2)

Head:


,icao,cluster
0,00AA,DENVER_CLUSTER
1,00AK,OTHER_CLUSTER
2,00AL,ATLANTA_CLUSTER
3,00AN,OTHER_CLUSTER
4,00AS,DALLAS_CLUSTER


In [26]:
# 1. Create the lookup dictionary using the correct column names
# Swapping 'ident' for 'icao' and 'Cluster' for 'cluster' based on your snippet
cluster_map = dict(zip(df_cluster['icao'], df_cluster['cluster']))

# 2. Map the individual clusters to your main flight 'df'
df['from_cluster'] = df['FromAirport'].map(cluster_map).fillna('Other')
df['to_cluster'] = df['ToAirport'].map(cluster_map).fillna('Other')

# 3. Create the concatenated corridor feature
df['From_To_Cluster'] = df['from_cluster'].astype(str) + ' - ' + df['to_cluster'].astype(str)

# 4. Keep that memory low! 
for col in ['from_cluster', 'to_cluster', 'From_To_Cluster']:
    df[col] = df[col].astype('category')

# 5. Verify the fix
print(df[['FromAirport', 'ToAirport', 'From_To_Cluster']].head())

  FromAirport ToAirport                              From_To_Cluster
0        KRIC      KENW      WASHINGTON_DC_CLUSTER - CHICAGO_CLUSTER
1        KSLC      KRIW  JACKSON_HOLE_CLUSTER - JACKSON_HOLE_CLUSTER
2        KEDC      KTVI            HOUSTON_CLUSTER - ATLANTA_CLUSTER
3        KBKS      KEDC            HOUSTON_CLUSTER - HOUSTON_CLUSTER
4        KEDC      KSRQ            HOUSTON_CLUSTER - ORLANDO_CLUSTER


In [30]:
# 1. Check for nulls in the main dataset
main_null_report = df.isnull().sum()

# 2. Show only columns that have at least one null value
print("Null values in main 'df':")
print(main_null_report[main_null_report > 0])

# 3. Get the percentage (often more useful for big datasets)
print("\nPercentage of missing data:")
print((main_null_report[main_null_report > 0] / len(df)) * 100)

Null values in main 'df':
Series([], dtype: int64)

Percentage of missing data:
Series([], dtype: float64)


In [32]:
# Check the final dimensions of your main dataframe
print(f"Main DataFrame Shape: {df.shape}")
print("-" * 30)

# Display the first few rows to see your new cluster columns
# Showing the last columns where your new features were added
df.head(10)

Main DataFrame Shape: (3636777, 21)
------------------------------


,FlightDate_utc,ArrivalDate_utc,Operator,FromAirport,FromCity,FromState,ToAirport,ToCity,ToState,Hours,...,aircraft_tailsign_certification,operator_type,aircraft_icao_code,aircraft_type,aircraft_model,aircraft_segment,fuel_uplift_usg,from_cluster,to_cluster,From_To_Cluster
0,2025-03-27 18:06:00+00:00,2025-03-27 19:54:00+00:00,Conquest Express LLC,KRIC,Richmond,VA,KENW,Kenosha,WI,1.800000,...,Part 91 / Non Commercial,Corporate Flight Department,LJ40,Learjet 40,Learjet 40-XR,Super Light Jet,363.60,WASHINGTON_DC_CLUSTER,CHICAGO_CLUSTER,WASHINGTON_DC_CLUSTER - CHICAGO_CLUSTER
1,2025-03-18 12:51:00+00:00,2025-03-18 13:40:00+00:00,MLA Management LLC,KSLC,Salt Lake City,UT,KRIW,Riverton,WY,0.816667,...,Part 91 / Non Commercial,Corporate Flight Department,C525,Cessna-Citation CJ1 / CitationJet / 525,Citation CJ1,Entry Level Jet,109.43,JACKSON_HOLE_CLUSTER,JACKSON_HOLE_CLUSTER,JACKSON_HOLE_CLUSTER - JACKSON_HOLE_CLUSTER
2,2025-01-26 20:28:00+00:00,2025-01-26 22:07:00+00:00,341HB LLC,KEDC,Austin,TX,KTVI,Thomasville,GA,1.650000,...,Part 91 / Non Commercial,Corporate Flight Department,LJ45,Learjet 45,Learjet 45,Super Light Jet,331.65,HOUSTON_CLUSTER,ATLANTA_CLUSTER,HOUSTON_CLUSTER - ATLANTA_CLUSTER
3,2025-02-16 21:16:00+00:00,2025-02-16 21:54:00+00:00,341HB LLC,KBKS,Falfurrias,TX,KEDC,Austin,TX,0.633333,...,Part 91 / Non Commercial,Corporate Flight Department,LJ45,Learjet 45,Learjet 45,Super Light Jet,127.30,HOUSTON_CLUSTER,HOUSTON_CLUSTER,HOUSTON_CLUSTER - HOUSTON_CLUSTER
4,2025-03-19 10:10:00+00:00,2025-03-19 12:11:00+00:00,341HB LLC,KEDC,Austin,TX,KSRQ,Sarasota Bradenton,FL,2.016667,...,Part 91 / Non Commercial,Corporate Flight Department,LJ45,Learjet 45,Learjet 45,Super Light Jet,405.35,HOUSTON_CLUSTER,ORLANDO_CLUSTER,HOUSTON_CLUSTER - ORLANDO_CLUSTER
5,2025-03-15 19:05:00+00:00,2025-03-15 19:33:00+00:00,G3 Aviation LLC,KHPN,White Plains,NY,KSFZ,Pawtucket,RI,0.466667,...,Part 91 / Non Commercial,Corporate Flight Department,SF50,Cirrus-SF-50 Vision,Vision SF50-G2+,Very Light Jet,32.20,NEW_YORK_CLUSTER,BOSTON_CLUSTER,NEW_YORK_CLUSTER - BOSTON_CLUSTER
6,2025-02-23 17:28:00+00:00,2025-02-23 18:49:00+00:00,N342ZK-Operator,KRSW,Fort Myers,FL,KCHA,Chattanooga,TN,1.350000,...,Part 91 / Non Commercial,Under Research,GLF6,Gulfstream-G600/650,G650,Ultra Long Range Jet,572.40,MIAMI_CLUSTER,ATLANTA_CLUSTER,MIAMI_CLUSTER - ATLANTA_CLUSTER
7,2025-02-25 22:10:00+00:00,2025-02-26 00:52:00+00:00,N342ZK-Operator,KBYH,Blytheville,AR,KSDL,Scottsdale,AZ,2.700000,...,Part 91 / Non Commercial,Under Research,GLF6,Gulfstream-G600/650,G650,Ultra Long Range Jet,1144.80,OTHER_CLUSTER,PHOENIX_CLUSTER,OTHER_CLUSTER - PHOENIX_CLUSTER
8,2025-02-01 00:01:00+00:00,2025-02-01 01:08:00+00:00,Franklin Mountain Assets II LLC,KDNA,Santa Teresa,NM,KRIL,Rifle,CO,1.116667,...,Part 91 / Non Commercial,Private Flight Department,GA6C,Gulfstream-G600/650,GVII-G600,Ultra Long Range Jet,473.47,Other,ASPEN_CLUSTER,Other - ASPEN_CLUSTER
9,2025-03-12 19:03:00+00:00,2025-03-12 21:40:00+00:00,CSMS Management LLC,KAUS,Austin,TX,KJAC,Jackson,WY,2.616667,...,Part 91 / Non Commercial,Corporate Flight Department,C700,Cessna-Citation Longitude,Citation Longitude,Super Midsize Jet,698.65,HOUSTON_CLUSTER,JACKSON_HOLE_CLUSTER,HOUSTON_CLUSTER - JACKSON_HOLE_CLUSTER


In [33]:
# 1. See all unique segments and how many flights are in each
# This helps you see where the "Bulk" of the market is
segment_counts = df['aircraft_segment'].value_counts()

print("Aircraft Segment Distribution:")
print(segment_counts)

# 2. Get just the list of unique names for your filter
unique_segments = df['aircraft_segment'].unique()
print("\nUnique Segments List:")
print(unique_segments)

Aircraft Segment Distribution:
aircraft_segment
Light Jet                 920509
Super Midsize Jet         820487
Heavy Jet                 490623
Ultra Long Range Jet      392215
Super Light Jet           377249
Midsize Jet               326085
Very Light Jet            200315
Entry Level Jet            89269
Airliner/Bizliner(Jet)     20025
Name: count, dtype: int64

Unique Segments List:
['Super Light Jet', 'Entry Level Jet', 'Very Light Jet', 'Ultra Long Range Jet', 'Super Midsize Jet', 'Light Jet', 'Midsize Jet', 'Heavy Jet', 'Airliner/Bizliner(Jet)']
Categories (9, object): ['Airliner/Bizliner(Jet)', 'Entry Level Jet', 'Heavy Jet', 'Light Jet', ..., 'Super Light Jet', 'Super Midsize Jet', 'Ultra Long Range Jet', 'Very Light Jet']


In [34]:
# 1. Filter for the specific segment
df_smj = df[df['aircraft_segment'] == 'Super Midsize Jet']

# 2. Check for the default values in geographic columns within this segment
unknown_from_state = (df_smj['FromState'] == 'International/Unknown').sum()
unknown_to_state = (df_smj['ToState'] == 'International/Unknown').sum()
unknown_from_city = (df_smj['FromCity'] == 'Unknown').sum()
unknown_to_city = (df_smj['ToCity'] == 'Unknown').sum()

print(f"--- Analysis for 'Super Midsize Jet' ({len(df_smj)} total flights) ---")
print(f"Flights from 'International/Unknown' State: {unknown_from_state}")
print(f"Flights to 'International/Unknown' State:   {unknown_to_state}")
print(f"Flights from 'Unknown' City:             {unknown_from_city}")
print(f"Flights to 'Unknown' City:               {unknown_to_city}")

# 3. Check if the segment itself has any 'Unknown' or 'Null' placeholders
print(f"\nNulls in segment column: {df['aircraft_segment'].isnull().sum()}")

--- Analysis for 'Super Midsize Jet' (820487 total flights) ---
Flights from 'International/Unknown' State: 121861
Flights to 'International/Unknown' State:   121859
Flights from 'Unknown' City:             4805
Flights to 'Unknown' City:               5023

Nulls in segment column: 0


In [35]:
import re

# 1. Define the "Wheels Up Family" keywords
# We use pipe | for 'OR' and (?i) for case-insensitivity
wu_pattern = r'(?i)wheels up|delta private|gama aviation|travel management|tmc jets|mountain aviation|grandview|alante air'

# 2. Check how many rows match this "Wheels Up Umbrella"
wu_mask = df['Operator'].str.contains(wu_pattern, na=False)
wu_count = wu_mask.sum()

print(f"Total flights identified under Wheels Up Umbrella: {wu_count}")
print(f"Percentage of total data: {(wu_count / len(df)) * 100:.2f}%")

# 3. See which specific operator names were caught
print("\nUnique Operator names found in this umbrella:")
print(df[wu_mask]['Operator'].unique())

Total flights identified under Wheels Up Umbrella: 51057
Percentage of total data: 1.40%

Unique Operator names found in this umbrella:
['Wheels Up Private Jets', 'Wheels Up Partners LLC', 'Mountain Aviation Leasing LLC', 'Grandview Jets LLC', 'Mountain Aviation', ..., 'Carlisle Travel Management Inc', 'Wheels Up Aviation LLC', 'Wheels Up Partners', 'Starr Mountain Aviation LLC', 'Gama Aviation FZC']
Length: 18
Categories (14898, object): ['01941-Operator', '01949-Operator', '0262 Aviation Inc Trustee', '09307-Operator', ..., 'flyNEAT', 'flydubai', 'iXAir', 'nextair LLC']


In [36]:
# 1. Define your logical conditions
is_short_hop = df['Hours'] <= 0.5
is_hub_shuffle = (df['from_cluster'] == df['to_cluster']) & \
                 (df['from_cluster'] != 'Other') & \
                 (df['Hours'] <= 1.0)

# 2. Create the feature: 'Y' for Reposition, 'N' for Revenue
# We use np.where for a fast, vectorized assignment
import numpy as np
df['reposition_flight'] = np.where(is_short_hop | is_hub_shuffle, 'Y', 'N')

# 3. Quick Vibe Check: See the breakdown
print("Repositioning Feature Breakdown:")
print(df['reposition_flight'].value_counts())

# 4. Verification: Average duration for each category
print("\nMean Flight Duration by Category:")
print(df.groupby('reposition_flight')['Hours'].mean())

Repositioning Feature Breakdown:
reposition_flight
N    2747938
Y     888839
Name: count, dtype: int64

Mean Flight Duration by Category:
reposition_flight
N    2.006193
Y    0.555681
Name: Hours, dtype: float32


In [39]:
# 1. The definitive Wheels Up Subsidiary List
wu_subsidiaries = [
    'Wheels Up Private Jets', 'Wheels Up Partners LLC', 'Wheels Up Aviation LLC', 
    'Wheels Up Partners', 'Mountain Aviation', 'Mountain Aviation Leasing LLC', 
    'Starr Mountain Aviation LLC', 'Grandview Jets LLC', 'GrandView Aviation', 
    'Gama Aviation LLC', 'Gama Aviation FZC', 'Travel Management Company LLC', 
    'TMC Jets', 'Carlisle Travel Management Inc', 'Delta Private Jets', 
    'Delta Private Jets LLC', 'Alante Air'
]

# 2. Create the boolean feature
df['is_wheels_up'] = df['Operator'].isin(wu_subsidiaries)

# 3. Filter for Super Midsize and group by the new feature
smj_check = df[df['aircraft_segment'] == 'Super Midsize Jet'].groupby('is_wheels_up').size()

# 4. MARKET SHARE LOGIC
# Extract the counts (using .get to avoid errors if a category is missing)
wu_smj_flights = smj_check.get(True, 0)
total_smj_flights = smj_check.sum()

# Calculate the percentage
if total_smj_flights > 0:
    smj_market_share = (wu_smj_flights / total_smj_flights) * 100
else:
    smj_market_share = 0

# 5. Final Output
print("--- Super Midsize Segment Analysis ---")
print(f"Wheels Up Flights: {wu_smj_flights:,}")
print(f"Total Market Size: {total_smj_flights:,}")
print(f"Wheels Up Market Share: {smj_market_share:.3f}%")

--- Super Midsize Segment Analysis ---
Wheels Up Flights: 12,332
Total Market Size: 820,487
Wheels Up Market Share: 1.503%
